# 01 - Exploratory Data Analysis

Profiling and statistics over the processed tables with PySpark, then the EDA
charts that motivate the modelling. Run the ingest + process stages first (or
`python -m tools.make_sample` for the sample).

In [ ]:
import sys
sys.path.append('..')
from src.common import get_spark, load_config, project_path
from pyspark.sql import functions as F

cfg = load_config()
spark = get_spark('eda')
pq = project_path(cfg['paths']['parquet'])
trip = spark.read.parquet(str(pq / 'fact_trip_delay'))
rbd = spark.read.parquet(str(pq / 'fact_route_band_day'))
print('trips', trip.count(), '| route-band-days', rbd.count())

## Profiling: nulls, cardinality, describe

In [ ]:
trip.describe(['median_delay', 'stops_observed']).show()
rbd.select(
    F.approx_count_distinct('route_id').alias('routes'),
    F.approx_count_distinct('time_band').alias('bands'),
    F.sum(F.col('compliant')).alias('compliant_rows'),
    F.count('*').alias('rows'),
).show()

## Statistics the brief names: mean, median, std, skewness, kurtosis

In [ ]:
trip.select(
    F.mean('median_delay').alias('mean'),
    F.expr('percentile_approx(median_delay, 0.5)').alias('median'),
    F.stddev('median_delay').alias('std'),
    F.skewness('median_delay').alias('skewness'),
    F.kurtosis('median_delay').alias('kurtosis'),
).show()

## Headline EDA chart: the long right tail of delay

In [ ]:
import matplotlib.pyplot as plt
pdf = trip.select('median_delay').toPandas()
pdf['median_delay'].clip(-10, 30).hist(bins=40)
plt.xlabel('trip median delay (min)'); plt.ylabel('trips'); plt.title('Delay distribution')
plt.show()

In [ ]:
# on-time rate by time band
rbd.groupBy('time_band').agg(F.avg('pct_on_time').alias('on_time')).orderBy('on_time').show()